# RAG Indexing Layer
### Ein geführter Workshop zur dritten Offline-Stufe: von Embeddings und Chunks zu prüfbaren Indizes

---

## Anschluss an die Embedding Layer

Die Embedding Layer ist abgeschlossen. Ihr Ergebnis sind zwei persistierte Dateien:

- `chunks.jsonl`: jeder Chunk mit `id`, `document_id`, `text` und `metadata`
- `embeddings.jsonl`: jeder Embedding-Record mit `id`, `chunk_id`, `document_id`, Vektor und Metadaten

Die Indexing Layer beginnt genau an dieser Grenze. Sie berechnet keine Embeddings,
lädt keine Rohdaten und führt kein Chunking aus. Sie verbindet bereits persistierte
Records und baut daraus die Datenstrukturen, die später von der Retrieval Layer genutzt werden.

```text
chunks.jsonl
embeddings.jsonl
    → Join über chunk_id
    → StoredDocument
    → DocumentStore
    → DenseIndex
    → SparseIndex
```

---

### Offline vs. Online

Auch dieses Notebook gehört vollständig zur Offline-Pipeline.

**Offline** bedeutet hier: Die Datenstrukturen werden vorab gebaut und persistiert.
Sie werden nicht pro Nutzerfrage neu erzeugt.

**Online** beginnt erst in der Retrieval Layer: Eine Query wird eingebettet, gegen
die Indizes gesucht, gerankt und anschließend mit Text aus dem DocumentStore verbunden.

---

### Warum Indexing eine eigene Layer ist

Embeddings allein reichen nicht für ein Retrieval-System. Eine Datei mit Vektoren
ist noch kein Index. Ohne Index müsste jede Query jeden gespeicherten Vektor einzeln
vergleichen.

Die Indexing Layer trennt deshalb drei Aufgaben:

**DenseIndex:** speichert Vektoren für semantische Suche.

**SparseIndex:** speichert tokenisierte Terme für lexikalische Suche.

**DocumentStore:** speichert Text und Metadaten als Single Source of Truth.

Diese Trennung ist entscheidend: Index-Backends bleiben textfrei, während der
DocumentStore die vollständigen Dokumentinformationen hält. Retrieval bekommt
später aus einem Index nur IDs und lädt den Text kontrolliert aus dem Store nach.

---

## 1. Input der Indexing Layer: persistierte Chunks und Embeddings

Die Indexing Layer hat genau zwei Inputs:

1. `chunks.jsonl` aus der Ingestion Layer
2. `embeddings.jsonl` aus der Embedding Layer

Wenn sie fehlen, erzeugt das Notebook kleine Demo-Daten mit derselben Struktur.
Dadurch bleibt der Workshop ausführbar, auch wenn die vorherigen Notebooks noch
nicht gelaufen sind.

In [ ]:
from pathlib import Path
import json
import logging
import math
import random

logging.getLogger("rag.indexing.orchestrator").setLevel(logging.ERROR)

embedding_store_path = Path("/home/jovyan/workspace/data/embeddings/embeddings.jsonl")
chunk_store_path     = Path("/home/jovyan/workspace/data/processed/chunks.jsonl")
index_base           = Path("/home/jovyan/workspace/data/index")

for label, path in [("chunks.jsonl", chunk_store_path), ("embeddings.jsonl", embedding_store_path)]:
    exists = path.exists()
    size   = f"{path.stat().st_size / 1024:.1f} KB" if exists else "–"
    print(f"{label:<18} exists={str(exists):<5} size={size}")

Die Daten werden als Listen materialisiert, weil dieses Notebook kleine,
nachvollziehbare Inspektionsschritte zeigt. Die produktive Architektur bleibt
trotzdem streamingfähig: `DocumentStore.stream_all()` und die Orchestrator-Schnittstellen
arbeiten mit Iterables.

In [ ]:
random.seed(42)

from rag.ingestion.storage.chunk_loader import load_chunks
from rag.embedding.store import EmbeddingStore


def make_demo_chunks(n=8):
    texts = [
        "Attention-Mechanismen gewichten relevante Tokens im Kontext.",
        "Transformer-Modelle bestehen aus Encoder- und Decoder-Bloecken.",
        "Retrieval-Augmented Generation kombiniert Suche mit Sprachmodellen.",
        "Dense Retrieval verwendet Vektorsimilaritaet zur Dokumentensuche.",
        "Sparse Retrieval basiert auf lexikalischer Uebereinstimmung von Termen.",
        "BM25 ist ein probabilistisches Rankingmodell fuer Textretrieval.",
        "FAISS ermoeglicht approximierte Nearest-Neighbor-Suche in Vektorraeumen.",
        "Der invertierte Index bildet Terme auf Dokument-IDs ab.",
    ]
    return [
        {
            "id":          f"chunk-{i:03d}",
            "document_id": f"doc-{i // 3:03d}",
            "text":        texts[i % len(texts)],
            "metadata":    {"source": f"doc_{i // 3:03d}.md", "chunk_index": i},
        }
        for i in range(n)
    ]

def make_demo_embeddings(chunks, dim=8):
    records = []
    for chunk in chunks:
        vector = [random.gauss(0, 1) for _ in range(dim)]
        norm   = math.sqrt(sum(x * x for x in vector))
        vector = [round(x / norm, 6) for x in vector]
        records.append({
            "id":             f"emb-{chunk['id']}",
            "chunk_id":       chunk["id"],
            "document_id":    chunk["document_id"],
            "embedding":      vector,
            "embedding_type": "dense",
            "metadata":       chunk["metadata"],
        })
    return records

# Echte Artefakte werden mit den Paket-Lesefunktionen geladen (Single Source of Truth).
# Fehlen sie, erzeugt das Notebook strukturgleiche Demo-Daten und bleibt lauffaehig.
if chunk_store_path.exists() and embedding_store_path.exists():
    raw_chunks     = list(load_chunks(chunk_store_path))
    raw_embeddings = EmbeddingStore(embedding_store_path).read_all()
    source_label   = "echte Daten"
else:
    raw_chunks     = make_demo_chunks()
    raw_embeddings = make_demo_embeddings(raw_chunks)
    source_label   = "Demo-Daten"

print(f"{source_label}: {len(raw_chunks)} Chunks | {len(raw_embeddings)} Embeddings")

---

## 2. Embedding-Records vorbereiten

Die beiden Inputs werden mit den **offiziellen Lese-Funktionen des `rag`-Pakets** geladen:
`load_chunks()` aus der Ingestion Layer und `EmbeddingStore.read_all()` aus der Embedding
Layer. Dadurch gibt es keine zweite, im Notebook nachgebaute JSONL-Leselogik. Die
Lese- und Validierungsregeln liegen ausschließlich im Paket.

Der `IndexingOrchestrator` verbindet anschließend Embeddings und Chunks über `chunk_id`
und liest den Dense-Vektor aus dem Feld `embedding`, also genau dem Feld, das die Embedding
Layer schreibt. Es ist deshalb **keine** Umbenennung oder Normalisierung der Records
nötig: Das Notebook reicht die Records unverändert an den Orchestrator weiter.

In [ ]:
# Keine Umbenennung noetig: der Orchestrator liest den Dense-Vektor aus 'embedding'.
embeddings = list(raw_embeddings)
chunks     = list(raw_chunks)

dense_vectors = [e["embedding"] for e in embeddings if e.get("embedding") is not None]
dense_dims    = {len(v) for v in dense_vectors}

print(f"Embedding-Records : {len(embeddings)}")
print(f"Dense-Vektoren    : {len(dense_vectors)}")
print(f"Dense-Dimensionen : {sorted(dense_dims)}")

Nach dieser Zelle gilt für den Rest des Notebooks: `embeddings` ist die indexierbare
Eingabe. `emb["embedding"]` enthält den Dense-Vektor, den der `IndexingOrchestrator`
später als `dense_vector` in den `StoredDocument` übernimmt.

---

## 3. Records inspizieren

Der Join der Indexing Layer basiert auf einer grundlegenden, aber kritischen Beziehung:

```python
embedding["chunk_id"] == chunk["id"]
```

Der Embedding-Record enthält den Vektor, aber keinen Text. Der Chunk-Record enthält
den Text, aber keinen Embedding-Vektor. Erst der Orchestrator verbindet beide zu
einem vollständigen `StoredDocument`.

In [ ]:
sample_embedding = embeddings[0]
sample_vector    = sample_embedding.get("embedding") or []

print(f"id             : {sample_embedding['id']}")
print(f"chunk_id       : {sample_embedding['chunk_id']}")
print(f"document_id    : {sample_embedding['document_id']}")
print(f"embedding      : dim={len(sample_vector)} preview={[round(x, 4) for x in sample_vector[:4]]}")
print(f"embedding_type : {sample_embedding.get('embedding_type')}")

In [ ]:
sample_chunk = chunks[0]

print(f"id          : {sample_chunk['id']}")
print(f"document_id : {sample_chunk['document_id']}")
print(f"text[:100]  : {sample_chunk['text'][:100]!r}")
print(f"metadata    : {sample_chunk.get('metadata', {})}")

Der Text bleibt bis zum Join ausschließlich im Chunk-Record. Das verhindert doppelte
Textspeicherung in Embedding-Dateien und macht den DocumentStore später zur
kontrollierten Quelle für Text und Metadaten.

---

## 4. Bereitschaft der Indexierung prüfen

Index-Aufbau ist eine schreibende Operation. Deshalb wird vor dem ersten Build
geprüft, ob die Eingaben konsistent sind:

- Embeddings und Chunks sind vorhanden
- IDs sind eindeutig
- jede `chunk_id` ist auflösbar
- Texte sind nicht leer
- Dense-Vektoren existieren und haben eine einheitliche Dimension

In [ ]:
chunk_ids      = [chunk["id"] for chunk in chunks]
embedding_ids  = [emb["id"] for emb in embeddings]
chunk_id_set   = set(chunk_ids)
unresolved     = [emb["chunk_id"] for emb in embeddings if emb.get("chunk_id") not in chunk_id_set]
dense_vectors  = [emb.get("embedding") for emb in embeddings if emb.get("embedding") is not None]
dense_dimensions = {len(vector) for vector in dense_vectors}

checks = {
    "Chunks vorhanden":               len(chunks) > 0,
    "Embeddings vorhanden":           len(embeddings) > 0,
    "Chunk-IDs eindeutig":            len(set(chunk_ids)) == len(chunks),
    "Embedding-IDs eindeutig":        len(set(embedding_ids)) == len(embeddings),
    "chunk_id vorhanden":             all(emb.get("chunk_id") for emb in embeddings),
    "alle chunk_ids aufloesbar":      len(unresolved) == 0,
    "Text nicht leer":                all(isinstance(chunk.get("text"), str) and chunk["text"].strip() for chunk in chunks),
    "Metadata ist Dict":              all(isinstance(chunk.get("metadata"), dict) for chunk in chunks),
    "Dense-Vektoren vorhanden":       len(dense_vectors) > 0,
    "Dense-Dimension einheitlich":    len(dense_dimensions) == 1,
    "Dense-Vektoren nicht leer":      all(len(vector) > 0 for vector in dense_vectors),
}

for label, ok in checks.items():
    print(f"[{'OK' if ok else 'FAIL'}] {label}")

if not all(checks.values()):
    raise RuntimeError("Indexing-Readiness fehlgeschlagen.")

dense_dim = next(iter(dense_dimensions))
print(f"INDEXING-READY | chunks={len(chunks)} embeddings={len(embeddings)} dim={dense_dim}")

---

## 5. Die zentralen Datentypen

Der Orchestrator trennt den vollständigen Dokumentzustand von den minimalen
Index-Eingaben.

| Typ | Enthält Text? | Enthält Dense-Vektor? | Zweck |
|---|---:|---:|---|
| `StoredDocument` | ja | optional | vollständiger Record im DocumentStore |
| `DenseIndexEntry` | nein | ja | minimale Eingabe für Dense-Backends |
| `TokenizedEntry` | nein | nein | minimale Eingabe für Sparse-Backends |
| `DenseCandidateResult` | nein | nein | Ergebnis einer Dense-Query: `id` + `score` |
| `SparseCandidate` | nein | nein | Ergebnis einer Sparse-Query: nur `id` |

Diese Typgrenzen sind die wichtigste Schutzschicht der Indexing Layer: Index-Backends
können keinen Text speichern, den sie nie erhalten.

In [ ]:
from rag.indexing.types import DenseIndexEntry, TokenizedEntry, StoredDocument

stored: StoredDocument = {
    "id":             "emb-demo",
    "chunk_id":       "chunk-demo",
    "document_id":    "doc-demo",
    "text":           "Dense und sparse Indizes speichern keinen Rohtext.",
    "dense_vector":   [0.1, 0.2, 0.3],
    "sparse_vector":  None,
    "embedding_type": "dense",
    "metadata":       {"source": "demo"},
}
dense_entry: DenseIndexEntry    = {"id": stored["id"], "dense_vector": stored["dense_vector"]}
tokenized_entry: TokenizedEntry = {"id": stored["id"], "tokens": ["dense", "sparse", "indizes"]}

for name, obj in [("StoredDocument", stored), ("DenseIndexEntry", dense_entry), ("TokenizedEntry", tokenized_entry)]:
    print(f"{name:<16} fields={len(obj):<2} text={'text' in obj} dense_vector={'dense_vector' in obj}")

Das Minimalprinzip ist nicht nur sauberer Code. Es verhindert Architekturfehler:
Ein FAISS-Index oder BruteForce-Index kann keinen Text versehentlich cachen,
weil die Eingabe nur aus `id` und Vektor besteht.

---

## 6. IndexConfig

`IndexConfig` beschreibt, welche Index-Strukturen gebaut werden.

```text
IndexConfig
  ├── mode: "dense" | "sparse" | "hybrid"
  ├── index_dir
  ├── dense: DenseIndexConfig
  │     ├── backend: "brute_force" | "faiss"
  │     ├── metric: "cosine" | "dot" | "l2"
  │     └── faiss: FAISSConfig
  └── sparse: SparseIndexConfig
        └── tokenizer: "simple" | "whitespace"
```

Alle Konfigurationsklassen sind `frozen=True`. Eine einmal erzeugte Konfiguration
kann nicht mehr verändert werden. Das macht Builds reproduzierbar.

In [ ]:
from rag.indexing.config import IndexConfig, DenseIndexConfig, SparseIndexConfig, FAISSConfig

config = IndexConfig(
    index_dir=index_base / "config_demo",
    mode="hybrid",
    dense=DenseIndexConfig(backend="brute_force", metric="cosine"),
    sparse=SparseIndexConfig(tokenizer="simple"),
)

print(f"mode      : {config.mode}")
print(f"backend   : {config.dense.backend}")
print(f"metric    : {config.dense.metric}")
print(f"tokenizer : {config.sparse.tokenizer}")

In [ ]:
try:
    config.mode = "dense"  # type: ignore
except Exception as e:
    print(f"frozen config      : {type(e).__name__}")

invalid_cases = [
    ("ungueltiger Modus",    lambda: IndexConfig(index_dir=index_base, mode="fuzzy")),
    ("ungueltige Metrik",    lambda: DenseIndexConfig(metric="hamming")),
    ("ungueltiger Backend",  lambda: DenseIndexConfig(backend="annoy")),
    ("ungueltiger Tokenizer", lambda: SparseIndexConfig(tokenizer="regex")),
]

for label, factory in invalid_cases:
    try:
        factory()
    except (ValueError, TypeError):
        print(f"{label:<22}: OK")

---

## 7. DocumentStore

Der `DocumentStore` ist die Single Source of Truth für Text und Metadaten.

Index-Backends liefern später nur IDs. Der Text wird danach über diese ID aus dem
DocumentStore nachgeladen. Deshalb muss jede ID, die in einem Dense- oder Sparse-Index
existiert, auch im DocumentStore existieren.

Der Orchestrator schreibt den DocumentStore bei jedem `build()` neu, unabhängig
vom `persist`-Flag. `persist=True` steuert nur, ob zusätzlich die Index-Backends
auf Disk geschrieben werden.

In [ ]:
from rag.indexing.store import DocumentStore

store_path = index_base / "store_demo" / "documents.jsonl"
doc_store  = DocumentStore(store_path)
doc_store.reset()

demo_docs = [
    {
        "id":             embeddings[i]["id"],
        "chunk_id":       chunks[i]["id"],
        "document_id":    chunks[i]["document_id"],
        "text":           chunks[i]["text"],
        "dense_vector":   embeddings[i].get("embedding"),
        "sparse_vector":  embeddings[i].get("sparse_embedding"),
        "embedding_type": embeddings[i].get("embedding_type", "dense"),
        "metadata":       chunks[i].get("metadata", {}),
    }
    for i in range(min(3, len(chunks), len(embeddings)))
]

doc_store.write_many(demo_docs)
print(f"StoredDocuments : {len(demo_docs)}")
print(f"Store size      : {store_path.stat().st_size} bytes")

Der Store bietet drei Lesepfade:

| Methode | Verhalten | Typischer Einsatz |
|---|---|---|
| `stream_all()` | liest Record für Record | große Bestände |
| `load_all()` | lädt alles als Liste | Inspektion |
| `load_by_id()` | lädt alles als Dict | Retrieval-Lookup nach Kandidaten-ID |

In [ ]:
first_doc  = next(doc_store.stream_all())
docs_by_id = doc_store.load_by_id()

print(f"stream_all id  : {first_doc['id']}")
print(f"text preview   : {first_doc['text'][:60]!r}")
print(f"load_by_id size: {len(docs_by_id)}")

---

## 8. IndexingOrchestrator: Join und Verteilung

Der `IndexingOrchestrator` ist die Laufzeitsteuerung der Indexing Layer.

Er macht drei Dinge:

1. Chunks validieren und als Map nach `id` verfügbar machen
2. Embeddings über `chunk_id` mit Chunks verbinden
3. Die entstandenen `StoredDocument`-Records auf DocumentStore, DenseIndex und
   SparseIndex verteilen

Was er bewusst nicht tut:

```text
Embedding-Berechnung  → rag.embedding
Chunking              → rag.ingestion
BM25-Scoring          → rag.retrieval
Hybrid-Fusion         → rag.retrieval
Reranking             → rag.retrieval
LLM-Inferenz          → außerhalb der Indexing Layer
```

In [ ]:
from rag.indexing.orchestrator import IndexingOrchestrator

dense_cfg = IndexConfig(
    index_dir=index_base / "dense_demo",
    mode="dense",
    dense=DenseIndexConfig(backend="brute_force", metric="cosine"),
    sparse=SparseIndexConfig(tokenizer="simple"),
)

dense_result = IndexingOrchestrator(dense_cfg).build(
    embeddings=iter(embeddings),
    chunks=iter(chunks),
    persist=False,
)

dense_docs    = list(dense_result.document_store.stream_all())
dense_doc_vecs = [doc["dense_vector"] for doc in dense_docs if doc.get("dense_vector")]

print(f"mode            : {dense_result.mode}")
print(f"documents       : {dense_result.document_count}")
print(f"dense backend   : {type(dense_result.dense_index).__name__}")
print(f"dense vectors   : {len(dense_doc_vecs)}")
print(f"dense dimension : {len(dense_doc_vecs[0])}")

In [ ]:
query_vec = dense_doc_vecs[0]
dense_top = dense_result.dense_index.query(query_vec, k=3)

for rank, item in enumerate(dense_top, start=1):
    print(f"{rank}. id={item['id'][:18]}... score={item['score']:+.5f}")

---

## 9. Modus: dense, sparse, hybrid

Der Modus steuert, welche Backends gebaut werden.

| Modus | DenseIndex | SparseIndex | Tokenizer |
|---|---:|---:|---:|
| `dense` | ja | nein | nein |
| `sparse` | nein | ja | ja |
| `hybrid` | ja | ja | ja |

Der Tokenizer wird nur instanziiert, wenn Sparse-Verarbeitung nötig ist.
Dense Retrieval braucht keine Tokenisierung.

In [ ]:
for mode in ["dense", "sparse", "hybrid"]:
    cfg = IndexConfig(
        index_dir=index_base / f"mode_{mode}",
        mode=mode,
        dense=DenseIndexConfig(backend="brute_force", metric="cosine"),
        sparse=SparseIndexConfig(tokenizer="simple"),
    )
    orchestrator  = IndexingOrchestrator(cfg)
    tok_state     = "yes" if orchestrator._tokenizer is not None else "no"
    print(f"mode={mode:<6} tokenizer={tok_state}")

---

## 10. Tokenisierung

Die Tokenisierung ist die wichtigste Invariante des Sparse-Pfads:

> Der Tokenizer zur Indexierungszeit muss derselbe sein wie zur Query-Zeit.

Der `SparseIndex` tokenisiert nie selbst. Er erhält nur `TokenizedEntry`-Records.
Dadurch ist klar: Text wird im Orchestrator zu Tokens, und Retrieval muss später
dieselbe Tokenizer-Konfiguration verwenden.

In [ ]:
from rag.indexing.sparse.tokenizer import Tokenizer

text = "BM25 ist ein TF-IDF-basiertes Rankingmodell (Robertson & Sparck Jones, 1976)."

tokenizer_simple     = Tokenizer(SparseIndexConfig(tokenizer="simple"))
tokenizer_whitespace = Tokenizer(SparseIndexConfig(tokenizer="whitespace"))

print(f"simple     : {tokenizer_simple.tokenize(text)}")
print(f"whitespace : {tokenizer_whitespace.tokenize(text)}")

`simple` extrahiert Wort-Tokens und entfernt Satzzeichen. `whitespace` trennt nur
an Leerzeichen und behält Satzzeichen im Token. Wenn Dokumente und Queries
unterschiedlich tokenisiert werden, entsteht Token-Drift.

---

## 11. Hybrid-Modus

Im Hybrid-Modus entstehen DenseIndex und SparseIndex aus denselben `StoredDocument`-Records.
Der DocumentStore wird trotzdem nur einmal geschrieben. Dense und sparse Backends sind
verschiedene Zugriffspfade auf dieselbe Dokumentbasis.

In [ ]:
hybrid_cfg = IndexConfig(
    index_dir=index_base / "hybrid_demo",
    mode="hybrid",
    dense=DenseIndexConfig(backend="brute_force", metric="cosine"),
    sparse=SparseIndexConfig(tokenizer="simple"),
)

hybrid_result = IndexingOrchestrator(hybrid_cfg).build(
    embeddings=iter(embeddings),
    chunks=iter(chunks),
    persist=False,
)

hybrid_docs = list(hybrid_result.document_store.stream_all())
hybrid_vecs = [doc["dense_vector"] for doc in hybrid_docs if doc.get("dense_vector")]

print(f"documents       : {hybrid_result.document_count}")
print(f"dense backend   : {type(hybrid_result.dense_index).__name__}")
print(f"sparse backend  : {type(hybrid_result.sparse_index).__name__}")
print(f"dense dimension : {len(hybrid_vecs[0])}")

---

## 12. BruteForceIndex

`BruteForceIndex` führt eine exakte Suche durch: Der Query-Vektor wird mit jedem
gespeicherten Vektor verglichen.

| Metrik | Bedeutung | Sortierung |
|---|---|---|
| `cosine` | Winkelähnlichkeit | höher ist besser |
| `dot` | Skalarprodukt | höher ist besser |
| `l2` | negative euklidische Distanz | höher ist besser |

Bei `l2` wird die Distanz negiert, damit ein identischer Vektor den höchsten Score hat: `0.0`.

In [ ]:
from rag.indexing.backends.brute_force import BruteForceIndex

entries = [
    {"id": emb["id"], "dense_vector": emb["embedding"]}
    for emb in embeddings
    if emb.get("embedding") is not None
]
query_vec = entries[0]["dense_vector"]

print(f"entries   : {len(entries)}")
print(f"dimension : {len(query_vec)}")

for metric in ["cosine", "dot", "l2"]:
    idx = BruteForceIndex(
        DenseIndexConfig(backend="brute_force", metric=metric),
        index_base / f"bf_{metric}",
    )
    idx.add(entries)
    top = idx.query(query_vec, k=1)[0]
    print(f"{metric:<6} score={top['score']:+.5f}")

In [ ]:
# Dimensionsfehler sind kontrollierte Fehler
dimension_cases = [
    (
        "config.dimension",
        lambda: BruteForceIndex(
            DenseIndexConfig(metric="cosine", dimension=4),
            index_base / "bf_dim_error",
        ).add([{"id": "x", "dense_vector": [0.1, 0.2, 0.3]}]),
    ),
    (
        "batch mismatch",
        lambda: (
            lambda idx: (
                idx.add([{"id": "a", "dense_vector": [0.1, 0.2, 0.3, 0.4]}]),
                idx.add([{"id": "b", "dense_vector": [0.1, 0.2, 0.3]}]),
            )
        )(BruteForceIndex(DenseIndexConfig(metric="cosine"), index_base / "bf_batch_error")),
    ),
]

for label, factory in dimension_cases:
    try:
        factory()
    except ValueError:
        print(f"{label:<18}: OK")

---

## 13. FAISSIndex

`FAISSIndex` implementiert dieselbe Dense-Backend-Schnittstelle wie `BruteForceIndex`.
Der Orchestrator muss deshalb nicht wissen, welches Backend verwendet wird.

FAISS bietet mehrere Index-Strukturen:

- `flat`: exakte Suche, kein Training
- `hnsw`: approximierte Suche, kein Training
- `ivf`: approximierte Suche mit Training

Für Cosine Similarity normalisiert die Implementierung Vektoren intern auf L2-Norm 1
und verwendet Inner Product.

In [ ]:
try:
    from rag.indexing.backends.faiss_index import FAISSIndex
    import faiss
    FAISS_AVAILABLE = True
    print(f"FAISS available: {faiss.__version__}")
except ImportError:
    FAISS_AVAILABLE = False
    print("FAISS available: no")

In [ ]:
if FAISS_AVAILABLE:
    faiss_flat = FAISSIndex(
        DenseIndexConfig(backend="faiss", metric="cosine", faiss=FAISSConfig(index_type="flat")),
        index_base / "faiss_flat_demo",
    )
    faiss_flat.add(entries)
    faiss_top = faiss_flat.query(query_vec, k=3)
    print(f"flat results : {len(faiss_top)}")
    print(f"top score    : {faiss_top[0]['score']:+.5f}")
else:
    print("skipped")

Die FAISS-Zelle ist optional: Wenn `faiss-cpu` nicht installiert ist, wird sie sauber
übersprungen. Das Notebook bleibt trotzdem vollständig ausführbar, weil
`BruteForceIndex` der stabile Dense-Backend für die didaktische Demonstration ist.

In [ ]:
if FAISS_AVAILABLE:
    ivf_case = lambda: FAISSIndex(
        DenseIndexConfig(
            backend="faiss", metric="cosine",
            faiss=FAISSConfig(index_type="ivf", ivf_nlist=10),
        ),
        index_base / "faiss_ivf_demo",
    ).add(entries)

    try:
        ivf_case()
        print("IVF training check: enough data")
    except ValueError:
        print("IVF training check: OK")
else:
    print("skipped")

---

## 14. SparseIndex

Der `SparseIndex` ist ein lexikalischer Kandidatenindex. Er scored nicht.

Das ist bewusst so: BM25 braucht vollständige Korpusstatistiken und ist query-abhängig.
Scoring gehört daher in die Retrieval Layer, nicht in die Indexing Layer.

`SparseCandidate` enthält deshalb nur:

```python
{"id": "..."}
```

Kein `score`-Feld. Dadurch kann Retrieval-Code nicht versehentlich einen falschen
oder uninformierten Score verwenden.

In [ ]:
sparse_index     = hybrid_result.sparse_index
query_tokens     = tokenizer_simple.tokenize("Attention Mechanismen relevante Tokens")
sparse_candidates = sparse_index.query(query_tokens, k=5)

print(f"query tokens : {query_tokens}")
print(f"candidates   : {len(sparse_candidates)}")
print(f"fields       : {list(sparse_candidates[0].keys()) if sparse_candidates else []}") 

---

## 15. InvertedIndexView

Der interne `InvertedIndex` speichert die eigentlichen Korpusstatistiken:

- Termfrequenzen
- Dokumentlängen
- Dokumentfrequenzen
- Vokabular

Externer Code greift darauf nicht direkt zu. Der erlaubte Zugang ist `sparse_index.get_view()`.
Diese View liefert genau die Informationen, die Retrieval später für BM25 oder TF-IDF braucht.

In [ ]:
view  = sparse_index.get_view()
stats = view.get_stats()
vocab = view.get_vocabulary()

print(f"doc_count      : {stats['doc_count']}")
print(f"avg_doc_length : {stats['avg_doc_length']:.2f}")
print(f"vocabulary     : {len(vocab)}")

In [ ]:
if sparse_candidates:
    candidate_id = sparse_candidates[0]["id"]
    for token in query_tokens[:3]:
        print(f"tf({token!r}) = {view.get_tf(token, candidate_id)}")
else:
    print("no candidates")

Das ist der Datenpfad für späteres BM25-Scoring: Retrieval erhält Kandidaten-IDs aus dem
SparseIndex, liest Korpusstatistiken über die View und berechnet den Score außerhalb
der Indexing Layer.

---

## 16. Persistenz und Load

Persistenz ist zweistufig:

| Ebene | Wann geschrieben | Datei |
|---|---|---|
| DocumentStore | immer bei `build()` | `documents.jsonl` |
| DenseIndex | bei `persist=True` | Backend-spezifisch |
| SparseIndex | bei `persist=True` | `dictionary.json`, `stats.json`, `vocabulary.json` |

Nach `load()` ist `document_count = -1`, weil der Orchestrator den DocumentStore
nicht automatisch scannt. Das ist Absicht: Laden soll schnell sein.

In [ ]:
persist_cfg = IndexConfig(
    index_dir=index_base / "persist_demo",
    mode="hybrid",
    dense=DenseIndexConfig(backend="brute_force", metric="cosine"),
    sparse=SparseIndexConfig(tokenizer="simple"),
)

persist_result = IndexingOrchestrator(persist_cfg).build(
    embeddings=iter(embeddings),
    chunks=iter(chunks),
    persist=True,
)

artifact_files = [p for p in sorted(persist_cfg.index_dir.rglob("*")) if p.is_file()]
for path in artifact_files:
    print(f"{path.relative_to(persist_cfg.index_dir)}")

In [ ]:
load_result = IndexingOrchestrator(persist_cfg).load()

build_top = [item["id"] for item in persist_result.dense_index.query(query_vec, k=5)]
load_top  = [item["id"] for item in load_result.dense_index.query(query_vec, k=5)]

print(f"loaded document_count : {load_result.document_count}")
print(f"dense roundtrip      : {build_top == load_top}")

---

## 17. Integritätsprüfung

Ein erfolgreicher Build ist nur dann nützlich, wenn DocumentStore, DenseIndex
und SparseIndex konsistent sind.

In [ ]:
docs     = list(persist_result.document_store.stream_all())
doc_ids  = {doc["id"] for doc in docs}
docs_with_dense = [doc for doc in docs if doc.get("dense_vector")]
doc_dims = {len(doc["dense_vector"]) for doc in docs_with_dense}

dense_results      = persist_result.dense_index.query(query_vec, k=5) if persist_result.dense_index else []
dense_ids_resolve  = all(item["id"] in doc_ids for item in dense_results)

sparse_stats  = persist_result.sparse_index.get_view().get_stats() if persist_result.sparse_index else None
sparse_ok     = sparse_stats is None or sparse_stats["doc_count"] == len(docs)

integrity_checks = {
    "DocumentStore nicht leer":     len(docs) > 0,
    "StoredDocument IDs eindeutig": len(doc_ids) == len(docs),
    "Texte nicht leer":             all(doc.get("text", "").strip() for doc in docs),
    "Dense-Vektoren vorhanden":     len(docs_with_dense) > 0,
    "Dense-Dimension einheitlich":  len(doc_dims) == 1,
    "Dense-IDs aufloesbar":         dense_ids_resolve,
    "Sparse-Stats konsistent":      sparse_ok,
}

for label, ok in integrity_checks.items():
    print(f"[{'OK' if ok else 'FAIL'}] {label}")

if not all(integrity_checks.values()):
    raise RuntimeError("Integritaetspruefung fehlgeschlagen.")

---

## 18. Bereitschaft des Index für die Retrieval Layer

Die Indexing Layer endet, wenn ein konsistenter, abfragbarer Zustand vorliegt.

Die Retrieval Layer erwartet:

1. einen DocumentStore mit Text und Metadaten
2. einen DenseIndex, der `DenseCandidateResult` liefert
3. einen SparseIndex, der `SparseCandidate` liefert
4. eine `InvertedIndexView` für BM25-Statistiken
5. auflösbare IDs zwischen Index-Backends und DocumentStore

In [ ]:
final      = persist_result
final_docs = list(final.document_store.stream_all())
final_ids  = {doc["id"] for doc in final_docs}

final_dense  = final.dense_index.query(query_vec, k=3) if final.dense_index else []
final_sparse = final.sparse_index.query(["test"], k=3) if final.sparse_index else []

final_dims  = {len(doc["dense_vector"]) for doc in final_docs if doc.get("dense_vector")}
final_dim   = next(iter(final_dims)) if final_dims else "n/a"
final_vocab = len(final.sparse_index.get_view().get_vocabulary()) if final.sparse_index else 0

readiness = {
    "DocumentStore bereit":           len(final_docs) > 0,
    "DenseIndex queryable":           final.dense_index is not None and isinstance(final_dense, list),
    "SparseIndex queryable":          final.sparse_index is not None and isinstance(final_sparse, list),
    "Dense-Dimension bekannt":        isinstance(final_dim, int),
    "Dense-Ergebnis IDs aufloesbar":  all(item["id"] in final_ids for item in final_dense),
}

for label, ok in readiness.items():
    print(f"[{'OK' if ok else 'FAIL'}] {label}")

if not all(readiness.values()):
    raise RuntimeError("Index-Readiness fehlgeschlagen.")

print()
print("INDEX-READY")
print(f"mode           : {final.mode}")
print(f"documents      : {len(final_docs)}")
print(f"dense backend  : {type(final.dense_index).__name__}")
print(f"dense dimension: {final_dim}")
print(f"sparse backend : {type(final.sparse_index).__name__}")
print(f"vocabulary     : {final_vocab}")
print(f"store          : {final.document_store.path}")

---

## 19. Zusammenfassung: Was die Indexing Layer leistet

---

### Was hier passiert ist

**Die Ingestion Layer hat `chunks.jsonl` erzeugt.**
Jeder Chunk enthält `id`, `document_id`, `text` und `metadata`.

**Die Embedding Layer hat `embeddings.jsonl` erzeugt.**
Jeder Embedding-Record enthält `id`, `chunk_id`, `document_id` und einen Vektor.

**Die Indexing Layer nutzt die Lese-Funktionen des Pakets.**
Chunks werden mit `load_chunks()`, Embeddings mit `EmbeddingStore.read_all()` geladen.
Der Orchestrator liest den Dense-Vektor direkt aus dem `embedding`-Feld, ohne
Umbenennung oder im Notebook nachgebaute Lese-Logik.

**Die Bereitschaftsprüfung prüft die Eingaben vor dem Build.**
Fehlende IDs, leere Texte, unauflösbare `chunk_id`s oder inkonsistente Vektordimensionen
werden vor dem ersten Schreibvorgang sichtbar.

**Der Orchestrator führt den Join durch.**
`embedding["chunk_id"]` wird mit `chunk["id"]` verbunden. Daraus entsteht `StoredDocument`.

**Der DocumentStore wird immer geschrieben.**
Er ist die Single Source of Truth für Text und Metadaten.

**DenseIndex und SparseIndex bleiben textfrei.**
Dense erhält nur `id + dense_vector`. Sparse erhält nur `id + tokens`.

**SparseIndex scored nicht.**
BM25, Fusion und Ranking gehören zur Retrieval Layer.

**Persistenz bildet die Systemgrenze.**
Nach `persist=True` liegen DocumentStore, DenseIndex und SparseIndex als Artefakte vor
und können später mit `load()` wiederhergestellt werden.

---

### Was die Indexing Layer bewusst nicht tut

```text
Embedding-Berechnung       → rag.embedding
Cleaning und Chunking      → rag.ingestion
BM25-Scoring               → rag.retrieval
Hybrid-Fusion              → rag.retrieval
Reranking                  → rag.retrieval
LLM-Inferenz               → außerhalb der Indexing Layer
```

---

### Datenfluss bis hierher

```text
Rohdaten
  → Ingestion Layer   → chunks.jsonl

chunks.jsonl
  → Embedding Layer   → embeddings.jsonl

chunks.jsonl + embeddings.jsonl
  → Join über chunk_id (Orchestrator)
  → Bereitschaftsprüfung
  → IndexingOrchestrator.build()
      → StoredDocument
      → DocumentStore
      → DenseIndexEntry → DenseIndex
      → TokenizedEntry  → SparseIndex
  → Persistenz
  → INDEX BEREIT

Nächste Stufe: Retrieval Layer
```